In [ ]:
!pip install pyspark -q
!pip install boto3 -q
!pip install pyarrow -q
!pip install databricks-sql-connector pandas -q

In [ ]:
from google.colab import drive
import os

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
    print("✅ Drive mounted successfully")
else:
    print("✅ Drive already mounted, skipping...")

✅ Drive already mounted, skipping...


In [ ]:
#configuration notebook

import os
os.chdir("/content/drive/My Drive/Colab Notebooks")
%run oo_config.ipynb

In [ ]:
import boto3
import os
import glob
import shutil
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from datetime import datetime
from pyspark.sql import DataFrame
from functools import reduce

# ── boto3 Client ──────────────────────────────────────────────────────────────
s3 = boto3.client(
    "s3",
    endpoint_url=G_MINIO_ENDPOINT,
    aws_access_key_id=G_MINIO_ACCESS_KEY,
    aws_secret_access_key=G_MINIO_SECRET_KEY
)

# ── Download All Partitioned Files from MinIO ─────────────────────────────────
local_dir  = "/tmp/CGS_Portfolio"
bucket     = "rawdatasets"
prefix     = "CGS_Portfolio/"

os.makedirs(local_dir, exist_ok=True)

response = s3.list_objects_v2(Bucket=bucket, Prefix=prefix)

for obj in response.get("Contents", []):
    s3_key = obj["Key"]

    # Skip if the key is just the folder prefix itself
    if s3_key.endswith("/"):
        continue

    # Reconstruct local path preserving year=/month= structure
    relative_path = os.path.relpath(s3_key, prefix)
    local_file_path = os.path.join(local_dir, relative_path)

    # Create subdirectories if needed (e.g. year=2023/month=6/)
    os.makedirs(os.path.dirname(local_file_path), exist_ok=True)

    s3.download_file(Bucket=bucket, Key=s3_key, Filename=local_file_path)
    #print(f"Downloaded: {s3_key} → {local_file_path}")

# ── Read All Partitions into Spark DataFrame ──────────────────────────────────
spark = SparkSession.builder \
    .appName("ReadCSGCSV") \
    .getOrCreate()

dfs = []

for filename in os.listdir(local_dir):
    if filename.endswith(".csv"):
        filepath = os.path.join(local_dir, filename)

        # Get file metadata
        modified_time = os.path.getmtime(filepath)
        date_modified = datetime.fromtimestamp(modified_time).strftime("%Y-%m-%d")

        # Read each CSV and append metadata columns
        df = spark.read \
            .option("header", "true") \
            .option("inferSchema", "true") \
            .csv(filepath)

        df = df.withColumn("source_file", F.lit(filename)) \
               .withColumn("date_modified", F.lit(date_modified))

        dfs.append(df)

# Union all DataFrames into one
final_df = reduce(DataFrame.union, dfs)

#final_df.printSchema()
#final_df.show()


In [ ]:
import re

def clean_column_names(df):
    new_columns = []
    for col in df.columns:
        # Remove special characters, keep only alphanumeric and underscores
        cleaned = re.sub(r'[^a-zA-Z0-9_]', '_', col)
        # Remove leading/trailing underscores
        cleaned = cleaned.strip('_')
        # Replace multiple consecutive underscores with single one
        cleaned = re.sub(r'_+', '_', cleaned)
        new_columns.append(cleaned)

    # Rename all columns
    for old, new in zip(df.columns, new_columns):
        df = df.withColumnRenamed(old, new)

    return df

# Apply to your DataFrame
final_df = clean_column_names(final_df)
final_df.printSchema()
final_df.show()

root
 |-- Symbol: string (nullable = true)
 |-- Code: string (nullable = true)
 |-- LACP: double (nullable = true)
 |-- Qty_Hand: integer (nullable = true)
 |-- Qty_Avai: integer (nullable = true)
 |-- Qty_Q_S: integer (nullable = true)
 |-- Avg_Buy_Prc: double (nullable = true)
 |-- Last: double (nullable = true)
 |-- Mkt_Val: double (nullable = true)
 |-- Un_G_L: double (nullable = true)
 |-- Chg: string (nullable = true)
 |-- Chg_Perc: string (nullable = true)
 |-- Yr_High: double (nullable = true)
 |-- Yr_Low: double (nullable = true)
 |-- Day_High: double (nullable = true)
 |-- Day_Low: double (nullable = true)
 |-- Vol: integer (nullable = true)
 |-- Lot_Size: integer (nullable = true)
 |-- Un_G_L_Perc: double (nullable = true)
 |-- Currency: string (nullable = true)
 |-- Exchg: string (nullable = true)
 |-- Qty_Sold: integer (nullable = true)
 |-- Qty_Susp: integer (nullable = true)
 |-- Bid: double (nullable = true)
 |-- Ask: double (nullable = true)
 |-- Avg_Sell_Price: double

In [ ]:
# ─── Databricks Connection Config ───────────────────────────────────────────
SERVER_HOSTNAME = G_SERVER_HOSTNAME
HTTP_PATH       = G_HTTP_PATH   # from Connection Details tab
ACCESS_TOKEN    = G_ACCESS_TOKEN         # replace with your PAT

# ─── Catalog / Schema / Table Target ────────────────────────────────────────
CATALOG   = "workspace"            # change to your catalog name (Unity Catalog)
SCHEMA    = "default"         # change to your schema/database
TABLE     = "cgs_portfolio"
FULL_TABLE = f"{CATALOG}.{SCHEMA}.{TABLE}"

In [ ]:
from databricks import sql

def run_query(cursor, query):
    """Helper to execute and optionally fetch results."""
    cursor.execute(query)
    try:
        return cursor.fetchall()
    except Exception:
        return None

COLUMNS = [
    "Symbol",
    "Code",
    "LACP",
    "Qty_Hand",
    "Qty_Avai",
    "Qty_Q_S",
    "Avg_Buy_Prc",
    "Last",
    "Mkt_Val",
    "Un_G_L",
    "Chg",
    "Chg_Perc",
    "Yr_High",
    "Yr_Low",
    "Day_High",
    "Day_Low",
    "Vol",
    "Lot_Size",
    "Un_G_L_Perc",
    "Currency",
    "Exchg",
    "Qty_Sold",
    "Qty_Susp",
    "Bid",
    "Ask",
    "Avg_Sell_Price",
    "source_file",
    "date_modified"
]

placeholders = ", ".join(["?"] * len(COLUMNS))
col_names    = ", ".join(COLUMNS)

with sql.connect(
    server_hostname = SERVER_HOSTNAME,
    http_path       = HTTP_PATH,
    access_token    = ACCESS_TOKEN
) as conn:
    with conn.cursor() as cursor:

        # ── Step 1: Set catalog and schema context ───────────────────────────
        cursor.execute(f"USE CATALOG {CATALOG}")
        cursor.execute(f"USE SCHEMA {SCHEMA}")

        # ── Step 2: Create table if not exists ───────────────────────────────
        create_sql = f"""
        CREATE TABLE IF NOT EXISTS {FULL_TABLE} (
              Symbol          VARCHAR(255),
              Code            VARCHAR(255),
              LACP            DOUBLE,
              Qty_Hand        INT,
              Qty_Avai        INT,
              Qty_Q_S         INT,
              Avg_Buy_Prc     DOUBLE,
              Last            DOUBLE,
              Mkt_Val         DOUBLE,
              Un_G_L          DOUBLE,
              Chg             VARCHAR(255),
              Chg_Perc        VARCHAR(255),
              Yr_High         DOUBLE,
              Yr_Low          DOUBLE,
              Day_High        DOUBLE,
              Day_Low         DOUBLE,
              Vol             INT,
              Lot_Size        INT,
              Un_G_L_Perc     DOUBLE,
              Currency        VARCHAR(255),
              Exchg           VARCHAR(255),
              Qty_Sold        INT,
              Qty_Susp        INT,
              Bid             DOUBLE,
              Ask             DOUBLE,
              Avg_Sell_Price  DOUBLE,
              source_file     VARCHAR(255)    NOT NULL,
              date_modified   VARCHAR(255)    NOT NULL
        )
        """
        cursor.execute(create_sql)
        print(f"✅ Table '{FULL_TABLE}' created/verified.")

        # ── Step 3: Insert rows from Pandas DataFrame ─────────────────────────
        #for _, row in pdf.iterrows():
        #    insert_sql = f"""
        #    INSERT INTO {FULL_TABLE} (id, name, score, is_active)
        #    VALUES ({row['id']}, '{row['name']}', {row['score']}, {str(row['is_active']).upper()})
        #    """
        #    cursor.execute(insert_sql)

        # Spark equivalent of: [tuple(r) for r in df.itertuples(index=False)]
        rows = [tuple(row) for row in df.collect()]

        #cursor.executemany(
        #    f"INSERT INTO {FULL_TABLE} VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)",
        #    rows
        #)

        insert_sql = f"""
        INSERT INTO {FULL_TABLE} ({col_names})
        SELECT {placeholders}
        WHERE NOT EXISTS (
            SELECT 1 FROM {FULL_TABLE}
            WHERE Symbol = ?
              AND Code = ?
              AND source_file = ?
        )
        """


        # Append the 3 key values at end of each row for the WHERE NOT EXISTS check
        rows_with_keys = [
            row + (row[COLUMNS.index("Symbol")],
                  row[COLUMNS.index("Code")],
                  row[COLUMNS.index("source_file")])
            for row in rows
        ]

        cursor.executemany(insert_sql, rows_with_keys)

        print(f"✅ Inserted {df.count()} rows into '{FULL_TABLE}'.")

        # ── Step 4: Verify by reading back ───────────────────────────────────
        results = run_query(cursor, f"SELECT * FROM {FULL_TABLE}")
        print(f"\n📦 Data in '{FULL_TABLE}':")
        for r in results:
            print(r)

✅ Table 'workspace.default.cgs_portfolio' created/verified.
✅ Inserted 98 rows into 'workspace.default.cgs_portfolio'.

📦 Data in 'workspace.default.cgs_portfolio':
Row(Symbol='Lion-OCBC Sec HSTECH S$.SI', Code='LIHS.SI', LACP=0.7929999828338623, Qty_Hand=1400, Qty_Avai=1400, Qty_Q_S=0, Avg_Buy_Prc=1.4019999504089355, Last=0.7860000133514404, Mkt_Val=1100.4000244140625, Un_G_L=-862.4000244140625, Chg='-0.007', Chg_Perc='-0.88', Yr_High=1.0839999914169312, Yr_Low=0.7179999947547913, Day_High=0.800000011920929, Day_Low=0.7850000262260437, Vol=3548747, Lot_Size=1, Un_G_L_Perc=-43.9370002746582, Currency='SGD', Exchg='SI', Qty_Sold=0, Qty_Susp=0, Bid=0.7860000133514404, Ask=0.7860000133514404, Avg_Sell_Price=0.0, source_file='20260303.csv', date_modified='2026-03-03')
Row(Symbol='Stoneweg EUTrust EUR.SI', Code='STON.SI', LACP=1.659999966621399, Qty_Hand=894, Qty_Avai=894, Qty_Q_S=0, Avg_Buy_Prc=0.0, Last=1.659999966621399, Mkt_Val=1484.0400390625, Un_G_L=0.0, Chg='0.000', Chg_Perc='0.00', 

In [ ]:
shutil.rmtree(local_dir, ignore_errors=True)
print(f"Deleted temp directory: {local_dir}")